# sparsetune 0.1.0: Google Colab GPU validation

This notebook validates the production PyPI package on one real NVIDIA GPU. It creates deterministic SPD sparse matrices, compares `scipy:cpu` and `cupy:cuda:0` using both `end-to-end` and `steady-state` measurements, checks the independently computed CPU residual evidence, and exports a JSON artifact.

> **Interpretation warning:** Colab GPU type, CPU allocation, clocks, CUDA image, and session lifetime vary. The results demonstrate functional behavior in this session; they are not a stable or cross-session performance baseline.

Select **Runtime > Change runtime type > T4 GPU** (or another NVIDIA GPU), then run all cells without editing them.

## 1. Detect CUDA and install one compatible CuPy build

GPU and CUDA information is collected before package selection. The cell fails early when no CUDA GPU is assigned. It removes conflicting CuPy distributions and installs `sparsetune==0.1.0` from production PyPI with exactly one of `cupy-cuda12x` or `cupy-cuda13x`.

In [ ]:
import re
import shutil
import subprocess
import sys

try:
    nvidia_smi = subprocess.check_output(["nvidia-smi"], text=True)
    gpu_query = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        text=True,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError(
        "No NVIDIA CUDA GPU is assigned. In Colab, select Runtime > Change "
        "runtime type > T4 GPU, then reconnect and run all cells again."
    ) from error

if not gpu_query:
    raise RuntimeError("nvidia-smi found no assigned NVIDIA GPU")

nvcc_output = ""
if shutil.which("nvcc"):
    nvcc_output = subprocess.check_output(["nvcc", "--version"], text=True)
runtime_match = re.search(r"release\s+(\d+\.\d+)", nvcc_output)
if runtime_match is None:
    runtime_match = re.search(r"CUDA Version:\s*(\d+\.\d+)", nvidia_smi)
if runtime_match is None:
    raise RuntimeError("Could not determine the CUDA runtime compatibility")

cuda_version = runtime_match.group(1)
cuda_major = int(cuda_version.split(".", maxsplit=1)[0])
cupy_packages = {12: "cupy-cuda12x", 13: "cupy-cuda13x"}
if cuda_major not in cupy_packages:
    raise RuntimeError(f"CUDA {cuda_major}.x is not supported by sparsetune 0.1.0")
cupy_package = cupy_packages[cuda_major]

print(f"Assigned GPU: {gpu_query}")
print(f"Detected CUDA compatibility: {cuda_version}")
print(f"Selected CuPy distribution: {cupy_package}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "cupy",
        "cupy-cuda11x",
        "cupy-cuda12x",
        "cupy-cuda13x",
    ],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "sparsetune==0.1.0",
        cupy_package,
    ],
    check=True,
)

## 2. Record and verify the environment

The checks below record `nvidia-smi`, Python, sparsetune, NumPy, SciPy, CuPy, and `sparsetune doctor`. A fresh subprocess must import the installed package and report `cupy:cuda:0`, which is the same import boundary used by isolated benchmark workers.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
import json
import platform

import cupy
import numpy
import scipy
import sparsetune

installed_cupy = {}
for package_name in ("cupy-cuda12x", "cupy-cuda13x"):
    try:
        installed_cupy[package_name] = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        pass
assert installed_cupy == {cupy_package: metadata.version(cupy_package)}, installed_cupy
assert sparsetune.__version__ == "0.1.0", sparsetune.__version__

version_output = subprocess.check_output(["sparsetune", "--version"], text=True).strip()
doctor_output = subprocess.check_output(
    ["sparsetune", "doctor", "--format", "json", "--quiet"], text=True
)
doctor = json.loads(doctor_output)
worker_backends = json.loads(
    subprocess.check_output(
        [
            sys.executable,
            "-c",
            "import json, sparsetune; print(json.dumps(sparsetune.list_backends()))",
        ],
        text=True,
    )
)
assert version_output.endswith("0.1.0"), version_output
assert "cupy:cuda:0" in worker_backends, worker_backends

environment = {
    "colab_runtime_date": datetime.now(timezone.utc).date().isoformat(),
    "python": platform.python_version(),
    "sparsetune": sparsetune.__version__,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "cupy": cupy.__version__,
    "cupy_distribution": installed_cupy,
    "gpu": gpu_query,
    "cuda_version": cuda_version,
    "cuda_major": cuda_major,
    "nvcc": nvcc_output.strip() or None,
    "nvidia_smi": nvidia_smi.strip(),
    "doctor": doctor,
    "subprocess_backends": worker_backends,
}
print(json.dumps(environment, indent=2))

## 3. Generate deterministic SPD sparse matrices

Each matrix is symmetric, strictly diagonally dominant, and has a positive diagonal. The fixed seed and visible parameters make the small, medium, and large cases reproducible without Google Drive or external data.

In [ ]:
from scipy.sparse import diags

SEED = 23
DTYPE = "float64"
MATRIX_SIZES = {"small": 10_000, "medium": 100_000, "large": 1_000_000}
RUNS = 3
RTOL = 1.0e-6
ATOL = 0.0
MAX_ITER = 10_000
TIMEOUT_SECONDS = 300.0


def make_spd_matrix(size, seed):
    rng = numpy.random.default_rng(seed)
    off_diagonal = rng.uniform(-0.25, 0.25, size - 1)
    diagonal = numpy.full(size, 2.0)
    diagonal[:-1] += numpy.abs(off_diagonal)
    diagonal[1:] += numpy.abs(off_diagonal)
    return diags(
        (off_diagonal, diagonal, off_diagonal),
        offsets=(-1, 0, 1),
        shape=(size, size),
        format="csr",
        dtype=DTYPE,
    )


matrices = {
    name: make_spd_matrix(size, SEED + index)
    for index, (name, size) in enumerate(MATRIX_SIZES.items())
}
[(name, matrix.shape, matrix.nnz) for name, matrix in matrices.items()]

## 4. Compare CPU and GPU and enforce numerical acceptance

Both backends run in isolated subprocesses with both measurement modes. sparsetune fetches each solution and computes the reported residual independently on the CPU. This cell rejects any backend that is not `converged`, has no relative residual, or exceeds its configured `convergence_threshold`.

In [ ]:
reports = {}
for name, matrix in matrices.items():
    print(f"Benchmarking {name}: shape={matrix.shape}, nnz={matrix.nnz}")
    report = sparsetune.benchmark(
        matrix,
        backends=("scipy:cpu", "cupy:cuda:0"),
        dtype=DTYPE,
        measure=("end-to-end", "steady-state"),
        runs=RUNS,
        rtol=RTOL,
        atol=ATOL,
        max_iter=MAX_ITER,
        timeout=TIMEOUT_SECONDS,
    )
    payload = report.to_dict()
    by_backend = {result["backend"]: result for result in payload["results"]}
    assert set(by_backend) == {"scipy:cpu", "cupy:cuda:0"}, by_backend
    for backend, result in by_backend.items():
        assert result["status"] == "converged", (name, backend, result)
        assert result["relative_residual"] is not None, (name, backend, result)
        assert result["relative_residual"] <= result["convergence_threshold"], (
            name,
            backend,
            result,
        )
    reports[name] = payload

## 5. Display recommendations and export JSON

The summary shows each recommendation's reason, speedup, and GPU break-even solve count when available. The artifact records environment details, tested GPU/CUDA/runtime date, dtype, matrix sizes and fingerprints, run parameters, complete results, and recommendations.

In [ ]:
for name, report in reports.items():
    print(f"\n{name.upper()} — {report['matrix']['fingerprint']}")
    for result in report["results"]:
        print(
            f"  {result['backend']}: status={result['status']}, "
            f"relative_residual={result['relative_residual']:.3e}, "
            f"threshold={result['convergence_threshold']:.3e}"
        )
    for mode, recommendation in report["recommendations"].items():
        print(
            f"  {mode}: backend={recommendation['backend']}, "
            f"reason={recommendation['reason']}, speedup={recommendation['speedup']}, "
            f"break_even_solves={recommendation['break_even_solves']}"
        )

parameters = {
    "seed": SEED,
    "dtype": DTYPE,
    "matrix_sizes": MATRIX_SIZES,
    "runs": RUNS,
    "rtol": RTOL,
    "atol": ATOL,
    "max_iterations": MAX_ITER,
    "timeout_seconds": TIMEOUT_SECONDS,
    "measurements": ["end-to-end", "steady-state"],
    "backends": ["scipy:cpu", "cupy:cuda:0"],
}
artifact = {
    "schema": "sparsetune-colab-gpu-validation-v1",
    "environment": environment,
    "parameters": parameters,
    "matrices": {
        name: {
            "shape": report["matrix"]["shape"],
            "nnz": report["matrix"]["nnz"],
            "matrix_fingerprint": report["matrix"]["fingerprint"],
        }
        for name, report in reports.items()
    },
    "results": reports,
}
artifact_path = "colab_gpu_validation_results.json"
with open(artifact_path, "w", encoding="utf-8") as artifact_file:
    json.dump(artifact, artifact_file, indent=2, sort_keys=True)
with open(artifact_path, encoding="utf-8") as artifact_file:
    json.load(artifact_file)
print(f"Wrote valid JSON artifact: {artifact_path}")

In [ ]:
from google.colab import files

files.download(artifact_path)

## Optional memory-pressure validation (disabled by default)

> **Warning:** Enable this only after downloading the default result artifact. It allocates a larger sparse matrix and can consume substantial host and GPU memory. Colab capacity varies, so this section intentionally applies moderate pressure and does **not** attempt to force a physical GPU OOM or destabilize the session.

In [ ]:
ENABLE_MEMORY_PRESSURE = False
MEMORY_PRESSURE_SIZE = 5_000_000

if ENABLE_MEMORY_PRESSURE:
    pressure_matrix = make_spd_matrix(MEMORY_PRESSURE_SIZE, SEED + 100)
    pressure_report = sparsetune.benchmark(
        pressure_matrix,
        backends=("scipy:cpu", "cupy:cuda:0"),
        dtype=DTYPE,
        measure=("end-to-end", "steady-state"),
        runs=1,
        rtol=RTOL,
        atol=ATOL,
        max_iter=MAX_ITER,
        timeout=TIMEOUT_SECONDS,
    )
    print(json.dumps(pressure_report.to_dict(), indent=2))
else:
    print("Skipped optional memory-pressure validation (safe default).")